In [1]:
# 임베딩 검색과 생성 모델 반복 실험에 필요한 패키지를 설치합니다.
# ==로 버전을 고정해 실패 코드의 API 형태를 일정하게 유지합니다.
%pip install -q "langchain==1.4.0" "langchain-openai==1.6.0" "python-dotenv==1.2.3"


Note: you may need to restart the kernel to use updated packages.


<h3>개념 요약</h3>
<p>같은 <code>찾을 수 없습니다</code>라는 답도 원인이 다를 수 있습니다. 문서 집합에 답이 없을 수도 있고, 답 문서가 있지만 top-k 검색에서 빠질 수도 있습니다. 먼저 검색 결과를 확인한 뒤 생성 응답을 읽어야 두 원인을 구분할 수 있습니다.</p>

In [2]:
# find_dotenv(usecwd=True)는 현재 작업 폴더에서 .env를 찾아 env_file에 경로를 반환합니다.
# load_dotenv()는 키를 환경 변수로 로드하고 os.getenv()는 OPENAI_API_KEY 존재를 확인합니다.
# assert는 자격 증명 준비가 되지 않았을 때 외부 호출 전에 실행을 멈춥니다.
import os

from dotenv import find_dotenv, load_dotenv

env_file = find_dotenv(usecwd=True)
assert env_file and load_dotenv(env_file), '프로젝트 루트 .env를 찾지 못했습니다.'
assert os.getenv('OPENAI_API_KEY'), 'OPENAI_API_KEY가 없습니다.'
print('환경 준비 완료')


환경 준비 완료


<h3>예제 실행</h3>
<p>질문과 단어가 많이 겹치지만 답이 없는 문서와, 실제 연체 규정이 있는 문서를 함께 색인합니다.</p>

In [3]:
# 첫 Document는 질문 단어가 많지만 규정 답이 없고, 둘째 Document는 실제 정답을 담습니다.
# from_documents()는 두 docs를 text-embedding-3-small로 임베딩해 store를 반환합니다.
# max_tokens=80은 각 생성 응답의 최대 길이를 제한하며 model은 반복 실험에 재사용합니다.
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain.chat_models import init_chat_model
from langchain_openai import OpenAIEmbeddings

docs = [Document(page_content='책을 3일 늦게 반납했습니다. 자세한 내용은 안내 데스크에 문의하세요.', metadata={'doc_id': 'DISTRACTOR'}),
        Document(page_content='자료를 늦게 반납하면 연체 일수만큼 대출이 정지됩니다.', metadata={'doc_id': 'ANSWER'})]
store = InMemoryVectorStore.from_documents(docs, OpenAIEmbeddings(model='text-embedding-3-small'))
model = init_chat_model('openai:gpt-5.4-mini', max_tokens=80)


<h3>top-k에 따른 검색 결과</h3>
<p>같은 질문을 <code>k=1</code>과 <code>k=2</code>로 검색하여 정답 문서가 문맥에 포함되는지 확인합니다.</p>

In [4]:
# late_question을 k=1과 k=2로 각각 검색해 one_hit과 two_hits에 Document 리스트를 받습니다.
# k는 반환할 상위 문서 개수이며, 리스트 표기식으로 각 결과의 doc_id를 확인합니다.
late_question = '책을 3일 늦게 반납하면 어떤 불이익이 있나요?'
one_hit = store.similarity_search(late_question, k=1)
two_hits = store.similarity_search(late_question, k=2)
print('k=1:', [d.metadata['doc_id'] for d in one_hit])
print('k=2:', [d.metadata['doc_id'] for d in two_hits])


k=1: ['DISTRACTOR']
k=2: ['DISTRACTOR', 'ANSWER']


<h3>두 프롬프트를 5회씩 실행</h3>
<p>답이 없는 A/S 기간 질문에 <code>helpful</code>과 <code>strict</code>를 각각 5회 적용합니다. 환각 횟수를 미리 가정하지 않고 실제 응답을 모두 기록합니다.</p>

In [5]:
# missing_context에는 질문의 답이 없으며 helpful과 strict는 같은 문맥을 다른 지시로 감쌉니다.
# 밖쪽 for는 (이름, 프롬프트) 튜플을 구조 분해하고, 리스트 표기식은 invoke()를 5회 호출합니다.
# enumerate(..., 1)은 각 반환 문자열에 1부터 시작하는 번호를 붙여 응답을 모두 출력합니다.
missing_context = '책 대출과 반납 규정만 있으며 A/S 처리 기간은 없습니다.'
question = '고장 난 기기의 A/S 처리 기간은 며칠인가요?'
helpful = f'문서를 참고해 친절하고 구체적으로 답하세요. 문서: {missing_context} 질문: {question}'
strict = f"문서에 답이 없으면 '찾을 수 없습니다'라고만 답하세요. 문서: {missing_context} 질문: {question}"
answers = {'helpful': [], 'strict': []}
for name, prompt in [('helpful', helpful), ('strict', strict)]:
    answers[name] = [model.invoke(prompt).text for _ in range(5)]
    for number, answer in enumerate(answers[name], 1):
        print(name, number, answer)


helpful 1 문서에는 **책 대출과 반납 규정만** 있고, **고장 난 기기의 A/S 처리 기간**에 대한 내용은 없습니다.  
따라서 **며칠인지 확인할 수 없습니다**.
helpful 2 문서에는 **책 대출과 반납 규정만** 있고, **고장 난 기기의 A/S 처리 기간에 대한 내용은 없습니다**.  
따라서 **A/S 처리 기간이 며칠인지 알 수 없습니다**.
helpful 3 문서에는 **책 대출과 반납 규정만** 있고, **고장 난 기기의 A/S 처리 기간에 대한 내용은 없습니다.**  
따라서 **A/S 처리 기간이 며칠인지 확인할 수 없습니다.**
helpful 4 문서에는 **책 대출과 반납 규정만** 있고, **A/S 처리 기간에 대한 내용은 없습니다**.  
따라서 **고장 난 기기의 A/S 처리 기간은 알 수 없습니다**.
helpful 5 문서에는 **책 대출과 반납 규정만** 있고, **A/S 처리 기간에 대한 내용은 없습니다**.  
따라서 **고장 난 기기의 A/S 처리 기간은 문서에서 확인할 수 없습니다**.


strict 1 찾을 수 없습니다
strict 2 찾을 수 없습니다
strict 3 찾을 수 없습니다
strict 4 찾을 수 없습니다
strict 5 찾을 수 없습니다


<h3>같은 거부 답의 두 원인</h3>
<p>첫 답은 문서 집합에 정보가 없어서, 둘째 답은 <code>k=1</code>이 정답 문서를 놓쳐서 발생합니다. 문자열이 같아도 검색 기록을 보면 원인이 다릅니다.</p>

In [6]:
# strict_answer()는 question과 Document 리스트 selected를 받고 응답 문자열을 반환합니다.
# join()과 제너레이터는 선택된 본문을 context 하나로 합치고, return은 AIMessage.text를 호출자에게 전달합니다.
# absent와 missed는 문서 집합의 답 부재와 top-k 검색 누락을 각각 재현한 응답입니다.
def strict_answer(question: str, selected: list[Document]) -> str:
    context = ' '.join(d.page_content for d in selected)
    prompt = f"문서에 답이 없으면 '찾을 수 없습니다'라고만 답하세요. 문서: {context} 질문: {question}"
    return model.invoke(prompt).text

absent = strict_answer(question, [Document(page_content=missing_context)])
missed = strict_answer(late_question, one_hit)
print('문서에 답 없음:', absent)
print('검색이 답을 놓침:', missed)


문서에 답 없음: 찾을 수 없습니다
검색이 답을 놓침: 찾을 수 없습니다


<h3>실행 검증</h3>
<p>반복 횟수와 검색 범위, 두 거부 답의 핵심 조건을 검사합니다.</p>

In [7]:
# len()으로 두 프롬프트가 각각 5회 실행되었는지, all()로 모든 응답이 비지 않았는지 검사합니다.
# 리스트와 집합 표기식은 k=1의 순서와 k=2의 전체 ID 구성을 각각 확인합니다.
# 마지막 assert의 and는 두 원인의 응답에 정해진 거부 문구가 모두 있어야 통과합니다.
assert len(answers['helpful']) == len(answers['strict']) == 5
assert all(answer for group in answers.values() for answer in group)
assert [d.metadata['doc_id'] for d in one_hit] == ['DISTRACTOR']
assert {d.metadata['doc_id'] for d in two_hits} == {'DISTRACTOR', 'ANSWER'}
assert '찾을 수 없습니다' in absent and '찾을 수 없습니다' in missed
print('검증 통과: 반복 응답과 서로 다른 두 실패 원인을 기록했습니다.')


검증 통과: 반복 응답과 서로 다른 두 실패 원인을 기록했습니다.


<h3>정리</h3>
<p><code>strict</code>는 근거가 없을 때의 응답 형식을 정하지만 검색 누락을 고치지는 못합니다. 생성 답, 검색 문서 ID, 문서 집합의 정답 존재 여부를 따로 기록해야 실패 원인을 구분할 수 있습니다.</p>

<p><strong>원인</strong> — <code>k=1</code>은 질문 단어가 많이 겹치는 문서만 가져와 정답 규정을 놓쳤습니다.</p>

In [8]:
# before_ids는 k=1 검색 결과 one_hit에서 doc_id만 추출한 리스트입니다.
# try의 assert는 ANSWER 존재를 기대하지만, except가 예상한 AssertionError를 포착합니다.
# missed도 함께 출력해 정답 문서 누락과 생성 응답을 연결해 봅니다.
before_ids = [d.metadata['doc_id'] for d in one_hit]
print('Before 검색:', before_ids)
print('Before 답:', missed)
try:
    assert 'ANSWER' in before_ids
except AssertionError:
    print('예상한 실패: 정답 문서가 문맥에 없습니다.')


Before 검색: ['DISTRACTOR']
Before 답: 찾을 수 없습니다
예상한 실패: 정답 문서가 문맥에 없습니다.


<p><strong>수정</strong> — <code>k</code>를 2로 늘려 정답 문서를 문맥에 포함한 뒤 같은 엄격한 프롬프트로 답합니다.</p>

In [9]:
# after_hits는 같은 질문의 k를 2로 늘려 다시 검색한 Document 리스트입니다.
# strict_answer()에 after_hits를 넘겨 수정된 문맥의 응답 문자열을 after_answer로 받습니다.
# 두 assert는 ANSWER 문서가 포함되었고 응답에 대출 정지 규정이 나타났는지 확인합니다.
after_hits = store.similarity_search(late_question, k=2)
after_ids = [d.metadata['doc_id'] for d in after_hits]
after_answer = strict_answer(late_question, after_hits)
print('After 검색:', after_ids)
print('After 답:', after_answer)
assert 'ANSWER' in after_ids
assert '대출' in after_answer and '정지' in after_answer


After 검색: ['DISTRACTOR', 'ANSWER']
After 답: 연체 일수만큼 대출이 정지됩니다.
